In [ ]:
import pandas as pd
import numpy as np
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil
import pytz  # ensure this is at the top of your script

root_folder = '../../../../../'
planning_data = os.path.join(root_folder,'Planning')
aware_folder = os.path.join(root_folder,'3_Analytics_Warehouse_Backend')

traffic_email_path = os.path.join(planning_data,'3_emails_from_traffic')
processed_path = os.path.join(traffic_email_path, 'processed')

phase2_sensor_path = os.path.join(planning_data,'4_phase_2_sensors/P2')


column_names_location = os.path.join(aware_folder,'1_inputs')#'../../../1_inputs'
parquet_intermediate_location = os.path.join(aware_folder,'2_planning_processing_files')#'../../../2_planning_processing_files'
database_location = os.path.join(aware_folder,'5_database')#'../../../5_database'

clear_folder = os.path.join(traffic_email_path, 'clear')

In [ ]:
# Columns
cols = pd.read_csv(os.path.join(column_names_location,"columns.csv"))["Column Names"].tolist()
cols_p2 = pd.read_csv(os.path.join(column_names_location,"p2_columns.csv"))["Column Names"].tolist()
sensors = pd.read_csv(os.path.join(planning_data,'Sensor_Mapping.csv'))
sensors

In [ ]:
df1 = pd.read_parquet(os.path.join(database_location,"database_cleaned_p1.parquet"), engine="pyarrow")
df2 = pd.read_parquet(os.path.join(database_location,"database_cleaned_p2.parquet"), engine="pyarrow")
dflong = pd.read_parquet(os.path.join(database_location,"database_cleaned_long.parquet"),engine = "pyarrow")
print("df1: ",len(df1))
print("df2: ",len(df2))
print("dflong: ",len(dflong))

In [ ]:
df1.tail()
dflong.tail()

In [ ]:
dflong.describe()

In [ ]:
files = [
    "WTC_Daily Archive P1_20251215_00-08-19.parquet",
    "WTC_Daily Archive P1_20251216_00-03-01.parquet",
    "WTC_Daily Archive P1_20251217_00-02-56.parquet",
    "WTC_Daily Archive P1_20251218_00-03-38.parquet",
    "WTC_Daily Archive P1_20251219_00-02-57.parquet",
    "WTC_Daily Archive P1_20251220_00-02-59.parquet",
    "WTC_Daily Archive P1_20251221_00-02-56.parquet",
    "WTC_Daily Archive P1_20251222_00-08-45.parquet",
    "WTC_Daily Archive P1_20251223_00-02-57.parquet",
    "WTC_Daily Archive P1_20251224_00-02-56.parquet",
    "WTC_Daily Archive P1_20251225_00-03-34.parquet",
    "WTC_Daily Archive P1_20251226_00-02-56.parquet",
    "WTC_Daily Archive P1_20251227_00-02-57.parquet",
    "WTC_Daily Archive P1_20251228_00-02-55.parquet",
    "WTC_Daily Archive P1_20251229_00-08-15.parquet",
    "WTC_Daily Archive P1_20251230_00-02-55.parquet",
    "WTC_Daily Archive P1_20251231_00-02-58.parquet",
]
base_path = os.path.join(
    parquet_intermediate_location,
    "p1_sensors_parquet",
    "processed"
)

dfs = []

for f in files:
    path = os.path.join(base_path, f)
    df = pd.read_parquet(path, engine="pyarrow")
    df["source_file"] = f  # helpful for QA/QC traceability
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

In [ ]:
df_all.describe()

In [ ]:
df_all.isna().sum().sum()

In [ ]:
schema_summary = []

for f in files:
    path = os.path.join(base_path, f)
    df = pd.read_parquet(path, engine="pyarrow")

    schema_summary.append({
        "source_file": f,
        "column_count": len(df.columns),
        "nas": df.isna().sum().sum()
    })

schema_df = pd.DataFrame(schema_summary)
schema_df.sort_values("source_file")



In [ ]:
sensor_cols = [c for c in df_all.columns if c.startswith("Z")]

def max_value_and_column(df):
    # max per column within the file
    col_max = df[sensor_cols].max()
    return pd.Series({
        "max_value": col_max.max(),
        "max_column": col_max.idxmax(),
        "nas": df.isna().sum().sum()
    })

result = (
    df_all
    .groupby("source_file")
    .apply(max_value_and_column)
    .sort_values("source_file", ascending=True)
)

result


In [ ]:
df = pd.read_excel(os.path.join(processed_path,"WTC_Daily Archive P1_20251219_00-02-57.xlsx"), sheet_name="Data", usecols=cols, engine="openpyxl")
df = df[df["From"].notnull()]
df = df[~df["From"].astype(str).str.contains("screenline list", case=False, na=False)]
df

In [ ]:
dflong.head()

In [ ]:
dflong.groupby("date").sum("Count")